# Assignment 2 - [student name]_[student ID]



# Preprocessing

In [7]:
# Firstly, we will do the preprocessing step for the diabetes MLP based on the given data.
# In this post, I use knowledge from this subject's Workshop Week 1 is "Python Basics with Jupyter Notebook", Week 2 is "Tensor Basics", Week 3 is "Autograd: Automatic Differentiation" and Week 4 is "Training a Classifier".
# Then, we will proceed to create the python libraries needed for this section.
import os
import json
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple
# Then, we use sklearn for preprocessing utilities.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import torch
# Step 1: We will do the path and directory.
# For the sake of detail, we will use a local project structure where ./data contains the input CSV files and all the output files, I do that so that your project can be zipped up and run simply and is easier to mark up and copy.
PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data"                 # input CSVs + all outputs live under ./data
EDA_DIR = DATA_DIR / "eda_figures"               # brief EDA images for the report
OUT_DIR = DATA_DIR / "processed"                 # serialized tensors + preprocessing artifacts
EDA_DIR.mkdir(parents=True, exist_ok=True)       # ensure folders exist
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Then, we will perform a move to the CSV file in the data folder.
TRAIN_CSV = DATA_DIR / "train_data.csv"
TEST_CSV  = DATA_DIR / "test_data.csv"
# Then we continue with the safety check, which we will do to help report errors quickly with clear messages if data files are missing.
assert TRAIN_CSV.exists(), f"Missing file: {TRAIN_CSV}"
assert TEST_CSV.exists(),  f"Missing file: {TEST_CSV}"
# Step 2: we proceed to load CSV.
# We'll read the CSV into Pandas DataFrames. We don't change the original data; we create a copy later. I prefer to keep the raw DataFrames as they are, which makes debugging easier and allows you to rerun steps independently.
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
# Then we do a quick visual display by printing out the shape and data type to confirm the basic schema; we display head() to visually check the columns/values, and we do validation checks and often uncover unexpected issues.
print(f"Train shape: {train_df.shape}")
print(f"Test shape : {test_df.shape}")
print("\nColumn dtypes (train):\n", train_df.dtypes, "\n")
display(train_df.head(10))
display(test_df.head(10))
# Step 3: we proceed to determine the target column.
# Then, we perform a search for common label names (Result/Goal/Label), and if none are found, we move on to detecting any binary numeric columns with exactly two unique values. This keeps the pipeline from being affected by small naming differences between groups.
candidate_names = ["Outcome", "outcome", "Target", "target", "Label", "label", "diabetes", "Diabetes"]
target_col = None
for name in candidate_names:
    if name in train_df.columns:
        target_col = name
        break
# Then, we perform redundancy, by scanning the columns to find the binary labels (two unique values).
if target_col is None:
    for col in train_df.columns:
        if not np.issubdtype(train_df[col].dtype, np.number):
            continue  # skip non-numeric columns
        uniq = pd.unique(train_df[col].dropna())
        if len(uniq) == 2:           # accept any binary set {0,1} or similar
            target_col = col
            break
# Then we do if it still isn't found, stop with a helpful message so you can fix the CSV file or rename the column.
if target_col is None:
    raise ValueError(
        "Could not infer the binary target column automatically. "
        "Please ensure the CSV contains a binary label column."
    )
print(f"Detected target column: {target_col!r}")
# Step 4: we perform the separation of features and targets.
# Then we perform the separation of objects (X) from labels (y) on the train set, reflecting the selection of objects on the test set.
feature_cols = [c for c in train_df.columns if c != target_col]
X_train_full = train_df[feature_cols].copy()
y_train_full = train_df[target_col].copy()
X_test_full  = test_df[feature_cols].copy()
# Step 5: we do basic EDA.
# We proceed to keep EDA concise by using class balancing, a histogram for the numerical features, and a simple correlation heatmap to show the interactions between the features.
# Step 5a: We perform class balance bar chart that helps decide whether we need class weighting or sampling.
class_counts = y_train_full.value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Class balance (train)")
plt.xlabel("Class label")
plt.ylabel("Count")
plt.tight_layout()
class_balance_path = EDA_DIR / "class_balance_train.png"
plt.savefig(class_balance_path)
plt.close()
# Step 5b: We do a characteristic (index) chart to check for distribution/outliers/zero inflation.
numeric_cols = [c for c in feature_cols if np.issubdtype(X_train_full[c].dtype, np.number)]
# We proceed to plot each numerical feature as a histogram on its own figure following the rules of graphing.
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    plt.hist(X_train_full[col].dropna(), bins=30)
    plt.title(f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    path = EDA_DIR / f"hist_{col}.png"
    plt.savefig(path)
    plt.close()
# Step 5c: We perform correlation heatmap (only numeric features) to flag multicollinearity before MLP.
if len(numeric_cols) > 1:
    corr = X_train_full[numeric_cols].corr().values
    plt.figure(figsize=(6,5))
    im = plt.imshow(corr, interpolation="nearest")
    plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
    plt.yticks(range(len(numeric_cols)), numeric_cols)
    plt.title("Feature Correlation (train)")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout()
    corr_path = EDA_DIR / "correlation_heatmap.png"
    plt.savefig(corr_path)
    plt.close()
# Step 6: We do the cleanup by imputing missing values.
# Then, we do it by using SimpleImputer(strategy='mean') on the numeric features and fitting to train, then we then convert both validation and test later. This avoids leaking test statistics into training. We see that if your dataset encodes impossible zeros (e.g. SkinThickness=0) as missing, you can replace the zeros with NaNs before imputing for specific columns and are retained here to preserve the original logic.
imputer = SimpleImputer(strategy="mean")
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_full[numeric_cols]), columns=numeric_cols)
X_test_imputed  = pd.DataFrame(imputer.transform(X_test_full[numeric_cols]), columns=numeric_cols)
# Then, we see that if there are any non-numeric columns (which is rare here), we remove them to keep the all-numeric MLP input data. This also records which columns were removed to ensure reproducibility.
dropped_non_numeric = [c for c in feature_cols if c not in numeric_cols]
if dropped_non_numeric:
    print(f"Dropping non-numeric columns (not used by MLP): {dropped_non_numeric}")
# Step 7: we proceed to de-correlation: remove highly correlated features.
# Then, we see that to stabilize the learning process and reduce overlap, remove one feature from each pair with |r|>0.95. We see that this is a conservative approach (only maximal collinearity). We apply each drop to check to keep the shapes aligned.
if X_train_imputed.shape[1] > 1:
    corr_matrix = X_train_imputed.corr().abs()
    to_drop = set()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if corr_matrix.iloc[i, j] > 0.95:
                to_drop.add(corr_matrix.columns[i])
    if to_drop:
        print(f"Removing highly correlated features (>|0.95|): {sorted(list(to_drop))}")
        X_train_imputed.drop(columns=list(to_drop), inplace=True)
        X_test_imputed.drop(columns=[c for c in to_drop if c in X_test_imputed.columns], inplace=True)
        numeric_cols = [c for c in numeric_cols if c not in to_drop]
# Step 8: We perform the training and validation split.
# Then, we retain a validation set (20%) from which to fine-tune hyperparameters and stop early. We perform label-wise stratification to maintain class balance between splits and it is important to get reliable metrics.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_imputed, y_train_full.values, test_size=0.2, random_state=42, stratify=y_train_full.values
)
# Step 9: we proceed to standardize the features.
# Then, we do just put StandardScaler into train split (X_tr), then apply to validation/test. This avoids leaking distribution statistics from validation/test into training.
scaler = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_te_scaled  = scaler.transform(X_test_imputed)
# Step 10: we proceed to convert to PyTorch tensor.
# Then, we do just convert numpy array to float32 tensor (features) and long tensor (labels) for CrossEntropyLoss. We keep test as feature only (no labels) to reflect the actual evaluation process.
X_tr_t  = torch.tensor(X_tr_scaled, dtype=torch.float32)
y_tr_t  = torch.tensor(y_tr, dtype=torch.long)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
X_te_t  = torch.tensor(X_te_scaled, dtype=torch.float32)
# Step 11: we save the artifacts for full reproducibility.
# Then we save the processed tensors and metadata so that the model code can load them without running EDA/cleaning again. Then we also serialize the loader and scaler to apply the same transformations later.
torch.save({"X": X_tr_t, "y": y_tr_t}, OUT_DIR / "train.pt")
torch.save({"X": X_val_t, "y": y_val_t}, OUT_DIR / "val.pt")
torch.save({"X": X_te_t}, OUT_DIR / "test.pt")
with open(OUT_DIR / "feature_names.json", "w") as f:
    json.dump(numeric_cols, f, indent=2)
with open(OUT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump({"imputer": imputer, "scaler": scaler}, f)
# Step 12: We do a final summary printout in this step.
# Then we do a printout of the shapes and file paths; this gives you a quick check for the report and allows the markers to track the results reported back to the code artifacts.
print("\n=== Preprocessing Summary ===")
print(f"Features used        : {len(numeric_cols)} -> {numeric_cols}")
print(f"Train tensor shape   : {tuple(X_tr_t.shape)}, labels: {tuple(y_tr_t.shape)}")
print(f"Val tensor shape     : {tuple(X_val_t.shape)}, labels: {tuple(y_val_t.shape)}")
print(f"Test tensor shape    : {tuple(X_te_t.shape)}")
print("\nArtifacts saved to:")
print(f" - {OUT_DIR / 'train.pt'}")
print(f" - {OUT_DIR / 'val.pt'}")
print(f" - {OUT_DIR / 'test.pt'}")
print(f" - {OUT_DIR / 'feature_names.json'}")
print(f" - {OUT_DIR / 'scaler.pkl'}")
print("\nEDA figures:")
print(f" - {EDA_DIR / 'class_balance_train.png'}")
# List one or two sample histogram files and the heatmap path if present.
hist_files = [f for f in os.listdir(EDA_DIR) if f.startswith('hist_')]
for hf in hist_files[:3]:                         # just preview a few histograms to avoid spamming the console
    print(f" - {EDA_DIR / hf}")
print(f" - {EDA_DIR / 'correlation_heatmap.png'}")

Train shape: (614, 9)
Test shape : (154, 9)

Column dtypes (train):
 Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object 



,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,98,58,33,190,34.0,0.430,43,0
1,2,112,75,32,0,35.7,0.148,21,0
2,2,108,64,0,0,30.8,0.158,21,0
3,8,107,80,0,0,24.6,0.856,34,0
4,7,136,90,0,0,29.9,0.210,50,0
5,6,103,72,32,190,37.7,0.324,55,0
6,1,71,48,18,76,20.4,0.323,22,0
7,0,117,0,0,0,33.8,0.932,44,0
8,4,154,72,29,126,31.3,0.338,37,0
9,5,147,78,0,0,33.7,0.218,65,0


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,1,85,66,29,0,26.6,0.351,31,0
1,0,137,40,35,168,43.1,2.288,33,1
2,1,189,60,23,846,30.1,0.398,59,1
3,5,166,72,19,175,25.8,0.587,51,1
4,3,126,88,41,235,39.3,0.704,27,0
5,8,99,84,0,0,35.4,0.388,50,0
6,1,97,66,15,140,23.2,0.487,22,0
7,3,88,58,11,54,24.8,0.267,22,0
8,10,122,78,31,0,27.6,0.512,45,0
9,4,103,60,33,192,24.0,0.966,33,0


Detected target column: 'Outcome'

=== Preprocessing Summary ===
Features used        : 8 -> ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Train tensor shape   : (491, 8), labels: (491,)
Val tensor shape     : (123, 8), labels: (123,)
Test tensor shape    : (154, 8)

Artifacts saved to:
 - C:\Users\Lenovo\OneDrive - University of Adelaide\Desktop\MASTER DEGREE\TRIMESTER 3 2025\Deep Learning Fundermentals\ASSIGNMENT 2\DLF Assignment 2\data\processed\train.pt
 - C:\Users\Lenovo\OneDrive - University of Adelaide\Desktop\MASTER DEGREE\TRIMESTER 3 2025\Deep Learning Fundermentals\ASSIGNMENT 2\DLF Assignment 2\data\processed\val.pt
 - C:\Users\Lenovo\OneDrive - University of Adelaide\Desktop\MASTER DEGREE\TRIMESTER 3 2025\Deep Learning Fundermentals\ASSIGNMENT 2\DLF Assignment 2\data\processed\test.pt
 - C:\Users\Lenovo\OneDrive - University of Adelaide\Desktop\MASTER DEGREE\TRIMESTER 3 2025\Deep Learning Fundermentals\ASSIG

# Model Implementation

In [8]:
# We will continue to the next part which is model implementation.
# In this exercise, I use knowledge from this subject's Workshop Week 1 is "Python Basics with Jupyter Notebook", Week 2 is "Tensor Basics", Week 3 is "Autograd: Automatic Differentiation" and Week 4 is "Training a Classifier".
# Firstly, we will create the python libraries required for this section.
import json
import math
import time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# Step 1: We proceed to prepare the copy and device.
# We continue to fix the seed error to reproduce and select the device (GPU if available, otherwise CPU).
def set_seed(seed: int = 42):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
# Step 2: We proceed to create the path and load the data.
# We continue to load the tensors saved during preprocessing; at the same time, read the feature name to find the metadata.
DATA_DIR = Path("data") / "processed"
train_blob = torch.load(DATA_DIR / "train.pt")
val_blob   = torch.load(DATA_DIR / "val.pt")
test_blob  = torch.load(DATA_DIR / "test.pt")
with open(Path("data") / "processed" / "feature_names.json") as f:
    FEATURE_NAMES = json.load(f)
# We proceed to divide the blocks into X/y; and from that deduce the input_dim and number of classes for the classifier.
X_tr, y_tr = train_blob["X"], train_blob["y"]
X_val, y_val = val_blob["X"], val_blob["y"]
X_te        = test_blob["X"]
input_dim = X_tr.shape[1]
num_classes = int(torch.max(y_tr).item() + 1)
print(f"Input dim: {input_dim}, Classes: {num_classes}")
# Step 3: We proceed to create Torch & DataLoader dataset.
# We continue to create the minimal dataset wrapper for the tensor (X, y); support the (y=None) check if needed.
class TensorDatasetXY(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor | None = None):
        self.X = X; self.y = y
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx):
        if self.y is None: return self.X[idx]
        return self.X[idx], self.y[idx]
# Then, we proceed to help build the loader with common defaults (batch size, on-board shuffling).
def make_loader(X, y=None, batch_size=64, shuffle=False):
    ds = TensorDatasetXY(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
# Step 4: We proceed to build the MLP model.
# We continue to create a simple configurable MLP model: Linear -> ReLU -> Dropout iteration, then Linear finally to logits.
class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int], out_dim: int, dropout: float = 0.2):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, out_dim)]       # no activation here; CrossEntropyLoss expects logits
        self.net = nn.Sequential(*layers)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
# Step 5: We proceed to the evaluation training utility.
# We continue to have the following: One training epoch: forward -> CE loss -> backward -> optimization step, accumulate average loss.
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader.dataset)
# We move on to the evaluation, where we calculate the average loss and standard classification metrics (accuracy/precision/recall/F1).
# Evaluate: returns loss + sklearn metrics.
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    all_y, all_p, total_loss = [], [], 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = model(Xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * Xb.size(0)
        preds = torch.argmax(logits, dim=1)
        all_y.append(yb.cpu()); all_p.append(preds.cpu())
    y_true = torch.cat(all_y).numpy()
    y_pred = torch.cat(all_p).numpy()
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary" if num_classes==2 else "macro", zero_division=0
    )
    return total_loss / len(loader.dataset), {"acc": acc, "prec": prec, "rec": rec, "f1": f1}
# Step 6: We proceed to the default hyperparameter configuration.
# We continue with reasonable default values ​​for tabular diabetes; will be changed later in the experiments.
@dataclass
class Config:
    hidden: List[int] = (64, 32)        # two hidden layers and adjust depth/width in experiments.
    dropout: float = 0.2                # dropout to combat overfitting on small tabular sets.
    lr: float = 1e-3                    # learning rate for Adam.
    weight_decay: float = 1e-4          # L2 regularization.
    batch_size: int = 64                # mini-batch size.
    epochs: int = 50                    # training epochs or max epochs; early stopping may end sooner.
    patience: int = 8                   # early stopping patience.
cfg = Config()
# Step 7: We proceed to build the model/loader/optimizer.
# We continue to create the loader, model, loss and optimizer. Thereby. we move the model to the device for acceleration.
train_loader = make_loader(X_tr, y_tr, batch_size=cfg.batch_size, shuffle=True)
val_loader   = make_loader(X_val, y_val, batch_size=cfg.batch_size, shuffle=False)
model = MLP(input_dim, list(cfg.hidden), num_classes, dropout=cfg.dropout).to(device)
criterion = nn.CrossEntropyLoss()    # CE expects raw logits and integer class labels.
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
# Step 8: We proceed to build the training loop with early stopping.
# Then, We continue to track the best validation F1; and through that we save the test point when improving, stopping after 'patience' does not improve.
BEST_PATH = Path("data") / "processed" / "best_model.pt"
best_f1, best_state, patience = -1.0, None, cfg.patience
for epoch in range(1, cfg.epochs + 1):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_metrics = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch:02d} | Training loss {tr_loss:.4f} | Validation loss {val_loss:.4f} "
          f"| Accuracy {val_metrics['acc']:.3f} | F1-score {val_metrics['f1']:.3f}")
# Then, we keep track of the latest F1 bests so we are robust against class imbalance versus accuracy alone.
    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_state = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": vars(cfg),
            "feature_names": FEATURE_NAMES,
            "input_dim": input_dim,
            "num_classes": num_classes,
        }
        torch.save(best_state, BEST_PATH)
        patience = cfg.patience  # reset patience when improved
    else:
        patience -= 1      # count down when no improvement
    if patience <= 0:
        print("Early stopping triggered.")     # prevent overfitting and save time
        break
print(f"\nBest Validation F1 score: {best_f1:.3f} | checkpoint saved to: {BEST_PATH}")

Device: cuda
Input dim: 8, Classes: 2
Epoch 01 | Training loss 0.7361 | Validation loss 0.6880 | Accuracy 0.488 | F1-score 0.547
Epoch 02 | Training loss 0.6834 | Validation loss 0.6434 | Accuracy 0.748 | F1-score 0.635
Epoch 03 | Training loss 0.6476 | Validation loss 0.6074 | Accuracy 0.780 | F1-score 0.620
Epoch 04 | Training loss 0.6202 | Validation loss 0.5738 | Accuracy 0.772 | F1-score 0.600
Epoch 05 | Training loss 0.5893 | Validation loss 0.5393 | Accuracy 0.772 | F1-score 0.611
Epoch 06 | Training loss 0.5669 | Validation loss 0.5055 | Accuracy 0.780 | F1-score 0.630
Epoch 07 | Training loss 0.5428 | Validation loss 0.4735 | Accuracy 0.805 | F1-score 0.684
Epoch 08 | Training loss 0.5216 | Validation loss 0.4494 | Accuracy 0.813 | F1-score 0.701
Epoch 09 | Training loss 0.5053 | Validation loss 0.4321 | Accuracy 0.813 | F1-score 0.701
Epoch 10 | Training loss 0.5095 | Validation loss 0.4213 | Accuracy 0.829 | F1-score 0.734
Epoch 11 | Training loss 0.4825 | Validation loss 0.

# Experiments

In [9]:
# We continue in this part is experiments.
# In this exercise, I use knowledge from this subject's Workshop Week 1 is "Python Basics with Jupyter Notebook", Week 2 is "Tensor Basics", Week 3 is "Autograd: Automatic Differentiation" and Week 4 is "Training a Classifier".
# Then, we will start by creating python libraries.
import json
import copy
import time
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List
import torch
import torch.nn as nn
# Step 1: We proceed to create the path and reproducibility.
# Next, we continue to create a test directory for logs and checkpoints; and then we start fixing the seed so that runs can be compared.
EXP_DIR = Path("data") / "processed" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)
set_seed(42)
# Step 2: We proceed to run and reload the processed tensors.
# We continue to reload the training/value/test tensors to keep this cell independent and reproducible.
DATA_DIR = Path("data") / "processed"
train_blob = torch.load(DATA_DIR / "train.pt")
val_blob   = torch.load(DATA_DIR / "val.pt")
test_blob  = torch.load(DATA_DIR / "test.pt")
with open(DATA_DIR / "feature_names.json") as f:
    FEATURE_NAMES = json.load(f)
# Then we proceed to decompress the tensor and the histogram, and from that we infer the input dimension and number of layers for the header.
X_tr, y_tr = train_blob["X"], train_blob["y"]
X_val, y_val = val_blob["X"], val_blob["y"]
X_te        = test_blob["X"]
input_dim   = X_tr.shape[1]
num_classes = int(torch.max(y_tr).item() + 1)
# Step 3: We proceed to create the experiment space.
# Then, we continue to define at least 3 separate settings; and then we change the depth/width, delay, LR and weight reduction.
@dataclass
class ExpCfg:
    name: str
    hidden: List[int]
    dropout: float
    lr: float
    weight_decay: float
    batch_size: int = 64
    max_epochs: int = 50
    patience: int = 8
# Then, we continue with creating three different configurations: base, wider and more spaced, shallower + lower LR/L2.
EXPS: List[ExpCfg] = [
    ExpCfg(name="A_baseline", hidden=[64, 32],   dropout=0.20, lr=1e-3,  weight_decay=1e-4),
    ExpCfg(name="B_wider_dp", hidden=[128, 64],  dropout=0.30, lr=1e-3,  weight_decay=1e-4),
    ExpCfg(name="C_shallow_lr", hidden=[32],     dropout=0.10, lr=5e-4,  weight_decay=5e-5),
# We can see that ExpCfg(name="D_deeper", hidden=[128,64,32], dropout=0.20, lr=1e-3,  weight_decay=1e-4).
]
# Step 4: We proceed to create the DataLoaders generator.
# Then, we continue to help a little to create the loaders; and then we just shuffle when training to randomize the batches.
def make_loader(X, y=None, batch_size=64, shuffle=False):
    ds = TensorDatasetXY(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
# Step 5: We proceed to train one experiment.
# Then, we continue training with an early stop at the F1 score value; and then we proceed to log the data in each epoch and save the best test score.
def run_experiment(cfg: ExpCfg):
# Then, we continue with building the loader according to the configuration (batch size, shuffle settings).
    train_loader = make_loader(X_tr, y_tr, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = make_loader(X_val, y_val, batch_size=cfg.batch_size, shuffle=False)
# Then, we continue with creating the model and optimizer and the specified gap; and from there we have CrossEntropy expecting logit and integer labels.
    model = MLP(input_dim, cfg.hidden, num_classes, dropout=cfg.dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
# Then, we do the bookkeeping to stop early and journalize.
    best_f1, best_state, patience = -1.0, None, cfg.patience
    rows = []  # collect per-epoch logs for a CSV
# Then, we do the standard train/validation loop with an early stop on the F1 score (more robust than accuracy).
    for epoch in range(1, cfg.max_epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_metrics = evaluate(model, val_loader, criterion)
# Then, we proceed to add the metrics for this epoch to our in-memory log.
        row = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": val_loss,
            "val_acc": val_metrics["acc"],
            "val_prec": val_metrics["prec"],
            "val_rec": val_metrics["rec"],
            "val_f1": val_metrics["f1"],
            "hidden": str(cfg.hidden),
            "dropout": cfg.dropout,
            "lr": cfg.lr,
            "weight_decay": cfg.weight_decay,
            "batch_size": cfg.batch_size,
        }
        rows.append(row)
        print(f"[{cfg.name}] Epoch {epoch:02d} | Training loss {tr_loss:.4f} | Validation loss {val_loss:.4f} | "
              f"Accuracy {val_metrics['acc']:.3f} | F1-score {val_metrics['f1']:.3f}")
# Then we see that the test score when F1 score improves; and we see that when we reset the patience, and we do keep a deep copy of the weight and preference state.
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = {
                "epoch": epoch,
                "model_state": copy.deepcopy(model.state_dict()),
                "optimizer_state": copy.deepcopy(optimizer.state_dict()),
                "feature_names": FEATURE_NAMES,
                "input_dim": input_dim,
                "num_classes": num_classes,
                "cfg": vars(cfg),
            }
            patience = cfg.patience  # reset when improved
        else:
            patience -= 1        # no improvement, count down
# Then, we stop early when there is no improvement in the patience phase to save time and avoid overfitting.
        if patience <= 0:
            print(f"[{cfg.name}] Early stopping.")
            break
# Then, we proceed to save the logs by epoch as CSV for tracking in reports.
    df = pd.DataFrame(rows)
    log_path = EXP_DIR / f"{cfg.name}_log.csv"
    df.to_csv(log_path, index=False)
# Then, we save the best checkpoint, in more detail the weights and configuration so that we can reuse and infer in the next steps.
    ckpt_path = EXP_DIR / f"{cfg.name}_best.pt"
    torch.save(best_state, ckpt_path)
# Then, we proceed to return the last row (for the trained epochs) and the best row (for the report summary).
    best_row = df.loc[df["val_f1"].idxmax()].to_dict()
    best_row.update({"ckpt_path": str(ckpt_path)})
    return df.iloc[-1].to_dict(), best_row
# Step 6: We proceed to run all experiments.
# Then, we continue to iterate the configurations; and then we collect the last and best rows to build the final summary tables.
last_rows, best_rows = [], []
for cfg in EXPS:
    last_row, best_row = run_experiment(cfg)
    last_row["name"] = cfg.name
    best_row["name"] = cfg.name
    last_rows.append(last_row)
    best_rows.append(best_row)
# Step 7: We proceed to create a summary table of results.
# Then, we continue to build concise tables, and we see that the best time for each experiment to rank and the final time to evaluate the reasonableness.
best_df = pd.DataFrame(best_rows)
last_df = pd.DataFrame(last_rows)
# Then, we proceed to rank the tests according to the F1 score (descending) to choose the winner in the report based on the data given by the question.
best_df_sorted = best_df.sort_values("val_f1", ascending=False).reset_index(drop=True)
# Then, we save the CSVs so that the grader can copy the exact data from our paper.
best_csv = EXP_DIR / "experiments_summary_best.csv"
last_csv = EXP_DIR / "experiments_summary_last.csv"
best_df_sorted.to_csv(best_csv, index=False)
last_df.to_csv(last_csv, index=False)
#print("\n=== Experiments Summary (Best Epoch per Run, sorted by F1 score) ===")
#display(best_df_sorted[[
    #"name", "epoch", "val_f1", "val_acc", "val_prec", "val_rec",
    #"hidden", "dropout", "lr", "weight_decay", "batch_size", "ckpt_path"
#]])
pretty = best_df_sorted.rename(columns={
    "val_f1": "validation_f1_score",
    "val_acc": "validation_accuracy",
    "val_prec": "validation_precision",
    "val_rec": "validation_recall",
    "hidden": "hidden_layer_sizes",
    "dropout": "dropout_rate",
    "lr": "learning_rate",
    "ckpt_path": "checkpoint_path",
})
print("\n=== Experiments Summary (Best Epoch per Run, sorted by F1-score) ===")
display(pretty[[
    "name", "epoch", "validation_f1_score", "validation_accuracy",
    "validation_precision", "validation_recall",
    "hidden_layer_sizes", "dropout_rate", "learning_rate",
    "weight_decay", "batch_size", "checkpoint_path"
]])
print("\nSaved summary CSVs:")
print(" -", best_csv)
print(" -", last_csv)
# Step 8: We proceed to create a test prediction with the top model.
# Then, we continue to create a test prediction from the best configuration checkpoint for completion.
TOP_NAME = best_df_sorted.iloc[0]["name"]
TOP_CKPT = best_df_sorted.iloc[0]["ckpt_path"]
# Then we proceed to load the checkpoint and run inference on the test set (feature only).
top_state = torch.load(TOP_CKPT, map_location=device)
top_model = MLP(input_dim, eval(json.loads(json.dumps(best_df_sorted.iloc[0]["hidden"]))) if isinstance(best_df_sorted.iloc[0]["hidden"], str) else best_df_sorted.iloc[0]["hidden"],
                num_classes, dropout=float(best_df_sorted.iloc[0]["dropout"])).to(device)
top_model.load_state_dict(top_state["model_state"])
top_model.eval()
# Then, we do inference on the test features; and then we output the labels and probabilities to the appendix if needed.
with torch.no_grad():
    logits = top_model(X_te.to(device))
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
# Then, we save the prediction for testing.
preds_df = pd.DataFrame({
    "pred_label": preds,
    "prob_class0": probs[:, 0],
    "prob_class1": probs[:, 1] if probs.shape[1] > 1 else 0.0,
})
preds_path = EXP_DIR / f"{TOP_NAME}_test_predictions.csv"
preds_df.to_csv(preds_path, index=False)
print(f"\nSaved test predictions for top run: {preds_path}")


[A_baseline] Epoch 01 | Training loss 0.7361 | Validation loss 0.6880 | Accuracy 0.488 | F1-score 0.547
[A_baseline] Epoch 02 | Training loss 0.6834 | Validation loss 0.6434 | Accuracy 0.748 | F1-score 0.635
[A_baseline] Epoch 03 | Training loss 0.6476 | Validation loss 0.6074 | Accuracy 0.780 | F1-score 0.620
[A_baseline] Epoch 04 | Training loss 0.6202 | Validation loss 0.5738 | Accuracy 0.772 | F1-score 0.600
[A_baseline] Epoch 05 | Training loss 0.5893 | Validation loss 0.5393 | Accuracy 0.772 | F1-score 0.611
[A_baseline] Epoch 06 | Training loss 0.5669 | Validation loss 0.5055 | Accuracy 0.780 | F1-score 0.630
[A_baseline] Epoch 07 | Training loss 0.5428 | Validation loss 0.4735 | Accuracy 0.805 | F1-score 0.684
[A_baseline] Epoch 08 | Training loss 0.5216 | Validation loss 0.4494 | Accuracy 0.813 | F1-score 0.701
[A_baseline] Epoch 09 | Training loss 0.5053 | Validation loss 0.4321 | Accuracy 0.813 | F1-score 0.701
[A_baseline] Epoch 10 | Training loss 0.5095 | Validation loss 0

,name,epoch,validation_f1_score,validation_accuracy,validation_precision,validation_recall,hidden_layer_sizes,dropout_rate,learning_rate,weight_decay,batch_size,checkpoint_path
0,B_wider_dp,5,0.743590,0.837398,0.805556,0.690476,"[128, 64]",0.3,0.0010,0.00010,64,data\processed\experiments\B_wider_dp_best.pt
1,A_baseline,10,0.734177,0.829268,0.783784,0.690476,"[64, 32]",0.2,0.0010,0.00010,64,data\processed\experiments\A_baseline_best.pt
2,C_shallow_lr,30,0.694444,0.821138,0.833333,0.595238,[32],0.1,0.0005,0.00005,64,data\processed\experiments\C_shallow_lr_best.pt



Saved summary CSVs:
 - data\processed\experiments\experiments_summary_best.csv
 - data\processed\experiments\experiments_summary_last.csv

Saved test predictions for top run: data\processed\experiments\B_wider_dp_test_predictions.csv
